# Windowed Data Loader — 30-Minute Intervals
Fetches data in 10-minute slices, groups them into 30-minute windows, and creates a Spark DataFrame per window. This prevents loading all ~9 lakh records into driver memory at once.

In [ ]:
import json
import requests
import urllib3
from datetime import datetime, timedelta
import pytz
from pyspark.sql import functions as F

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

In [ ]:
# ── Configuration ────────────────────────────────────────────────────────────
file_date   = "2026-06-02"
url         = "<YOUR_ELASTICSEARCH_URL>/_search"
user        = "<USERNAME>"
psd         = "<PASSWORD>"
headers     = {"Content-Type": "application/json"}

all_scopes  = ['SWCC', 'TDCC', 'PBCC', 'PTCC']

kolkata_tz  = pytz.timezone("Asia/Kolkata")

INTERVAL_MIN  = 10   # minutes per API fetch
WINDOW_MIN    = 30   # minutes per DataFrame window
INTERVALS_PER_WINDOW = WINDOW_MIN // INTERVAL_MIN   # = 3

total_intervals = (24 * 60) // INTERVAL_MIN         # = 144

In [ ]:
def fetch_interval(start_time: str, end_time: str) -> list:
    """Fetch one 10-minute slice from Elasticsearch. Returns list of hit dicts."""
    response = requests.post(
        url,
        headers=headers,
        auth=(user, psd),
        json={
            "size": 10000,
            "fields": ["*"],
            "query": {
                "bool": {
                    "must": [
                        {"range": {"@timestamp": {"gte": start_time, "lte": end_time}}},
                        {"terms": {"Scope.keyword": all_scopes}},
                    ]
                }
            },
        },
        verify=False,
        timeout=120,
    )
    if response.status_code != 200:
        raise Exception(f"API call failed for interval {start_time} - {end_time}: {response.text}")
    return response.json().get("hits", {}).get("hits", [])


def flatten_hits(hits: list) -> list:
    """Flatten Elasticsearch hit['fields'] into plain dicts."""
    records = []
    for record in hits:
        flat = {}
        for key, value in record.get("fields", {}).items():
            flat[key] = value[0] if isinstance(value, list) and len(value) > 0 else value
        records.append(flat)
    return records


def create_df(flattened_records: list):
    """Create a Spark DataFrame from flattened records."""
    if not flattened_records:
        return None
    return spark.createDataFrame(flattened_records)

In [ ]:
# ── Main loop — process one 30-minute window at a time ───────────────────────
grand_total = 0
total_windows = total_intervals // INTERVALS_PER_WINDOW  # 48 windows

for window_idx in range(total_windows):
    window_hits   = []   # cleared after every window — keeps peak memory low
    window_start  = window_idx * WINDOW_MIN
    window_end    = window_start + WINDOW_MIN - 1

    win_start_h, win_start_m = divmod(window_start, 60)
    win_end_h,   win_end_m   = divmod(window_end,   60)
    win_label = (
        f"{file_date}T{win_start_h:02d}:{win_start_m:02d}:00"
        f" → {file_date}T{win_end_h:02d}:{win_end_m:02d}:59"
    )
    print(f"\n── Window {window_idx + 1}/{total_windows}: {win_label}")

    # Fetch INTERVALS_PER_WINDOW consecutive 10-minute slices
    for step in range(INTERVALS_PER_WINDOW):
        interval_idx    = window_idx * INTERVALS_PER_WINDOW + step
        start_total_min = interval_idx * INTERVAL_MIN
        end_total_min   = start_total_min + INTERVAL_MIN - 1

        start_h, start_m = divmod(start_total_min, 60)
        end_h,   end_m   = divmod(end_total_min,   60)

        start_time = f"{file_date}T{start_h:02d}:{start_m:02d}:00.000"
        end_time   = f"{file_date}T{end_h:02d}:{end_m:02d}:59.999"

        hits = fetch_interval(start_time, end_time)
        window_hits.extend(hits)
        print(
            f"  interval {interval_idx + 1}/{total_intervals} "
            f"({start_time[-12:-4]} → {end_time[-12:-4]}): "
            f"{len(hits):,} records | window subtotal: {len(window_hits):,}"
        )

    grand_total += len(window_hits)

    # ── Create DataFrame for this window only ────────────────────────────────
    flattened = flatten_hits(window_hits)
    df = create_df(flattened)

    if df is not None:
        print(f"  DataFrame: {df.count():,} rows, {len(df.columns)} columns")

        # ── Place your per-window transformations / writes here ───────────────
        # Example: df.write.mode("append").parquet(f"/mnt/output/{file_date}/window_{window_idx+1:02d}")
        # ─────────────────────────────────────────────────────────────────────

    # Release memory before the next window
    del window_hits, flattened, df

print(f"\nAll {total_windows} windows processed. Grand total records: {grand_total:,}")